# 0 - EDA (Visualisation des données, remise en contexte)

Via l'EDA on pourrait voir que :
- "age" a 50 valeurs aberrantes (age > 500 ans)
- "cholesterol" est fortement corrélé avec la target (r=0.82)
- "profession = Boulanger" → risque moyen 2x plus élevé
- "vitamin_D" a 30% de valeurs manquantes

Donc on pourra ajuster le preprocessing en conséquence
```python
def preprocess(X):
    X = remove_outliers(X, 'age', threshold=200)  # Grâce à l'EDA
    X = impute_missing(X, 'vitamin_D', strategy='median')  # Grâce à l'EDA
    # ...
```

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set_theme()   # active le style seaborn pour matplotlib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, LabelEncoder

import matplotlib.pyplot as plt

## 1. Structure des données
Combien de schtroumpf ? combien de feature ? quel type de données ? valeurs manquantes ? doublons ? 

In [ ]:
X_train = pd.read_csv('../data/data_labeled/X_train.csv')
y_train = pd.read_csv('../data/data_labeled/y_train.csv')
X_test = pd.read_csv('../data/data_labeled/X_test.csv')
y_test = pd.read_csv('../data/data_labeled/y_test.csv')
X_unlabeled = pd.read_csv('../data/data_unlabeled/X.csv')

print(f"Shape X_train: {X_train.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_test: {y_test.shape}")
print(f"Shape X_unlabeled: {X_unlabeled.shape}")

**Visualisations:**

In [ ]:
print("\nValeurs manquantes:")
print(X_train.isnull().sum())

X_train.describe()

## 2. Quel target ?
- Quelle est la distribution du risque cardiaque ?
- La distribution est-elle normale ? Skewed ? 
- Quel est le range des valeurs ?
- Y a-t-il d'extreme genre "outliers" ?

In [ ]:
plt.figure(figsize=(10, 6))
y_train.hist(bins= 50)
plt.title('Distribution du risque cardiaque')

plt.figure(figsize=(8, 6))
sns.boxplot(y=y_train.values.flatten())
plt.title('Boxplot pour mieux voir les outliers')
plt.show()

## 3. Feature numérique
### Questions
- Quelles features ont des outliers (donnée aberrante) ?
- Y a-t-il des corrélations entre features ? 
- Quelle feature est la plus corrélée avec la target (le risque de faire un arret cardiaque) ?
- Y a-t-il de la multicolinéarité ? (2 features trop similaires)

### Réponses

- Je sais pas ?
- Oui, il y a des corrélations entre weight et height, cholesterol et blood pressure, blood press et weight, weight et cholesterol
- Vitamine D: Plus de vitamine D -> Moins de risque (corrélation négative) (je pensais que ca allait etre le cholesterol mais au final peut-etre que c'est pas une correlation lineaire (on verra dans l'analyse non lineaire))
- Non, pour l'instant je pense que le seuil max c'est 0.85 de correlation et on est à la moitié là.

In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = X_train[numeric_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Matrice de corrélation entre features numériques')

X_train_with_target = X_train.copy()
X_train_with_target['target'] = y_train
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
correlations_with_target = X_train_with_target[numeric_features + ['target']].corr()['target']
correlations_with_target = correlations_with_target.drop('target') # Pas avoir la correlation avec elle meme 
correlations_with_target = correlations_with_target.sort_values(key=abs, ascending=False)

print(correlations_with_target)

# Sélectionner les 6 features les plus corrélées
top_features = correlations_with_target.head(9).index.tolist()
print(f"Top 6 features les plus corrélées : {top_features}")

fig, axes = plt.subplots(3, 3, figsize=(18, 10))
fig.suptitle('Scatter Plots : Top Features vs Target (Risque Cardiaque)', fontsize=16)
for i, feature in enumerate(top_features):
    ax = axes[i//3, i%3]
    
    # pcq il y a une différence de longueur
    n = min(len(X_train), len(y_train))
    ax.scatter(X_train[feature].iloc[:n], y_train.iloc[:n].squeeze(), alpha=0.5, s=20)
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Risk (Target)', fontsize=12)
    
    corr_value = correlations_with_target[feature]  
    ax.set_title(f'{feature} (r={corr_value:.3f})', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Analyse categorical feature

### Questions :
- Combien de catégories dans "profession" ?
- Les distributions sont-elles équilibrées ?
- Y a-t-il des catégories rares (< 1% des données) ?
- Quel impact sur la target ?

### Réponses :
- 6 catégories de professions
- ?
- Smallest est à 8% donc ok
- ?

In [ ]:
smallest_pourcentage = X_train['profession'].value_counts().min()/X_train['profession'].value_counts().sum()
smallest = X_train['profession'].min()
print(f"Pourcentage of the smallest : {smallest} -> {smallest_pourcentage}")

plt.figure(figsize=(12,6))
X_train['profession'].value_counts().plot(kind='bar')
plt.title('Distribution des professions')
plt.xticks(rotation=45)
plt.show()

risk_by_profession = X_train.copy()
risk_by_profession['target'] = y_train
risk_by_profession.groupby('profession')['target'].mean().sort_values().plot(kind='barh')
plt.title('Risque moyen par profession')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ordinal_features = ['sarsaparilla', 'smurfberry liquor', 'smurfin donuts']
for i, feature in enumerate(ordinal_features):
    X_train[feature].value_counts().sort_index().plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Distribution: {feature}')
plt.show()


## 5. Anomalies
### Questions 
- Y a-t-il des valeurs impossibles ? (âge négatif, poids = 0)
- Y a-t-il des patterns évidents ? (ex: cholestérol ↑ → risque ↑)
- Y a-t-il des sous-groupes distincts ? (clusters)

In [ ]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for i, col in enumerate(numeric_cols):
    ax = axes[i//3, i%3]
    sns.boxplot(data=X_train, y=col, ax=ax)
    ax.set_title(f'Détections des outliers - {col}')
plt.tight_layout()
plt.show()

sns.pairplot(X_train_with_target[['age', 'cholesterol', 'blood pressure','vitamin D', 'target']], 
             hue='target', diag_kind='kde')
plt.show()